# Colab GPU로 공 검출 모델 학습하기

`smart-factory-dual-arm`의 YOLO 검출 모델을 Colab GPU에서 학습한다.

## 이 노트북의 설계

| 무엇을 | 어디에 | 왜 |
|---|---|---|
| **체크포인트** | Google Drive에 **직접** | Colab은 예고 없이 끊기고 그때 `/content`는 전부 사라진다. `output_dir`을 Drive로 잡으면 `last.pt`가 매 epoch Drive에 쌓인다 |
| **학습 데이터** | 로컬 디스크(`/content`) | Drive FUSE는 파일 하나당 지연이 커서 3천 장을 매 epoch 읽으면 GPU가 논다. 압축 하나를 Drive에 두고 로컬로 풀어 쓴다 |
| **COCO 혼합분** | Colab에서 재생성 | `coco_others` 세션은 원본(COCO val2017, 약 1GB)이 커서 압축에 넣지 않는다. 4b단계가 직접 받아 같은 세션을 재현한다(시드 고정) |

**학습 셀은 몇 번이든 다시 실행해도 된다.** `resume=True`가 체크포인트가 있으면 이어서, 없으면 처음부터 시작한다.

## 시작 전 준비 (로컬 PC에서 한 번)

데이터셋을 압축해 Drive에 올린다. COCO 관련 폴더는 뺀다 — 원본(`coco_src`)은
1GB가 넘고, 생성 세션(`coco_others`)은 4b단계가 Colab에서 다시 만든다.

```bash
cd ~/Documents/smart-factory-dual-arm
tar --exclude=train_set/coco_src --exclude=train_set/coco_others \
    -czf train_set.tar.gz train_set
```

만들어진 `train_set.tar.gz`를 Google Drive의 **`MyDrive/smart-factory-dual-arm/`** 폴더에 업로드한다.

> ⚠️ 지금 데이터는 PNG 1280×960 3015장 = **3.8GB**다. 업로드가 부담되면 JPEG로 바꾸면 용량이 크게 줄고 학습 결과 차이는 거의 없다. 라벨 `.txt`는 확장자와 무관하게 **파일명**으로 매칭되므로 건드릴 필요가 없다.
>
> ```python
> from pathlib import Path
> from PIL import Image
> for png in Path("train_set").rglob("*.png"):
>     Image.open(png).convert("RGB").save(png.with_suffix(".jpg"), quality=92)
>     png.unlink()
> ```

## 1. GPU 확인

**런타임 → 런타임 유형 변경 → 하드웨어 가속기: GPU** 로 설정돼 있어야 한다.
`CUDA True`가 안 나오면 아래를 진행해도 CPU로 돌아 매우 느리다.

In [ ]:
!nvidia-smi

import torch

print("torch :", torch.__version__)
print("CUDA  :", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU 없음 — 런타임 유형을 확인하세요")

## 2. 저장소 클론(또는 최신화) + ultralytics 설치

`train_set/`과 `models/`는 `.gitignore` 대상이라 클론에 포함되지 않는다. 각각 4·6단계에서 채운다.

이미 클론돼 있으면 **원격의 최신 커밋으로 맞춘다.** 로컬 PC에서 코드를 고쳐 푸시한 뒤
이 셀을 다시 실행하면 그대로 반영된다. `train_set/`은 추적 대상이 아니라 그대로 남는다.

In [ ]:
from pathlib import Path

REPO = Path("/content/smart-factory-dual-arm")
BRANCH = "feat/yolo"
REMOTE = "https://github.com/yolo-dual-arm-assembly/smart-factory-dual-arm.git"

if REPO.exists():
    # 얕은 클론(--depth 1)이라 git pull은 까다롭다. fetch 후 그 지점으로 맞춘다.
    !git -C {REPO} fetch --depth 1 origin {BRANCH} && git -C {REPO} reset --hard FETCH_HEAD
else:
    !git clone --branch {BRANCH} --depth 1 {REMOTE} {REPO}

!git -C {REPO} log --oneline -1

%pip install -q ultralytics==8.4.102
# Colab 기본 torch가 요구하는 것보다 낡은 sympy가 남아 있으면 학습 시작 때
# "module 'sympy' has no attribute 'printing'"으로 죽는다. 미리 맞춰 둔다.
# (이미 낡은 sympy가 import된 뒤라면 '런타임 → 세션 다시 시작' 후 재실행.)
%pip install -q -U sympy

## 3. Google Drive 마운트

인증 팝업이 뜨면 계정을 선택하고 접근을 허용한다.
`vision_inspection.colab`이 마운트와 폴더 생성을 담당하며, 이미 마운트돼 있으면 다시 붙지 않는다.

In [ ]:
import sys

for sub in ("ros2_ws/src/common", "ros2_ws/src/vision_inspection"):
    path = str(REPO / sub)
    if path not in sys.path:
        sys.path.insert(0, path)

# 2번 셀에서 코드를 최신화했다면 이미 import한 낡은 모듈이 메모리에 남아 있다.
# 지워 두면 런타임을 다시 시작하지 않아도 새 파일을 읽는다.
for name in [m for m in sys.modules if m.split(".")[0] in ("vision_inspection", "common")]:
    del sys.modules[name]

from vision_inspection import colab

DRIVE = colab.drive_dir()        # MyDrive/smart-factory-dual-arm/
RUNS = colab.drive_runs_dir()    # MyDrive/smart-factory-dual-arm/runs/  ← 체크포인트가 여기 쌓인다

print("Drive 프로젝트 :", DRIVE)
print("학습 결과 저장 :", RUNS)

## 4. 데이터셋 준비 (Drive → 로컬 디스크)

Drive에 올려 둔 압축을 저장소 루트로 푼다. 압축 안에 `train_set/`이 최상위로 들어 있어서
`REPO/train_set/...`이 되고, 이는 `data.yaml`이 기대하는 위치와 정확히 같다.

이미 풀려 있으면 건너뛴다 — 셀을 다시 실행해도 4GB를 다시 풀지 않는다.

In [ ]:
ARCHIVE = DRIVE / "train_set.tar.gz"

colab.extract_dataset(ARCHIVE, REPO)

!ls -1 {REPO}/train_set

## 4b. COCO 혼합 세션 만들기 (coco_others)

`data.yaml`의 train 목록에는 COCO에서 뽑은 `coco_others` 세션이 들어 있다
(sports ball→ball, 나머지 물체 전부→others 재매핑 — 오투입 일반화용).
원본(val2017)이 약 1GB라 tar.gz에는 넣지 않고, **여기서 직접 받아 재생성한다.**
`coco_import`의 시드가 고정이라 로컬 PC에서 만든 것과 같은 400장이 나온다.

Colab 회선에서 다운로드는 1~2분. 이미 만들어져 있으면 건너뛴다.
(런타임이 초기화되면 `/content`가 비워지므로 다시 받는다.)

In [ ]:
COCO_SRC = REPO / "train_set" / "coco_src"
COCO_OUT = REPO / "train_set" / "coco_others"

if (COCO_OUT / "images" / "train").is_dir() and any((COCO_OUT / "images" / "train").iterdir()):
    print("[coco] coco_others가 이미 있어 건너뜁니다.")
else:
    if not (COCO_SRC / "annotations" / "instances_val2017.json").is_file():
        COCO_SRC.mkdir(parents=True, exist_ok=True)
        !cd {COCO_SRC} && wget -q http://images.cocodataset.org/zips/val2017.zip && unzip -q -o val2017.zip && rm val2017.zip
        !cd {COCO_SRC} && wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip && unzip -q -o annotations_trainval2017.zip && rm annotations_trainval2017.zip

    from vision_inspection.coco_import import CocoImportConfig, import_coco

    summary = import_coco(
        CocoImportConfig(
            annotations=COCO_SRC / "annotations" / "instances_val2017.json",
            images_dir=COCO_SRC / "val2017",
            output_dir=COCO_OUT,
        )
    )
    print(
        f"[coco] images={summary.images} ball={summary.ball_instances}"
        f" others={summary.others_instances}"
    )

    # /content 디스크를 아끼고 싶으면 원본은 지워도 된다 — 세션은 이미 만들어졌다.
    # !rm -rf {COCO_SRC}

## 5. 데이터셋 검증

학습을 시작하기 전에 경로와 라벨 매칭이 맞는지 확인한다.
여기서 이미지 수가 0이면 압축 구조가 잘못된 것이다.

In [ ]:
import glob, os
from ultralytics.data.utils import check_det_dataset, img2label_paths

DATA_YAML = REPO / "ros2_ws/src/vision_inspection/config/data.yaml"
info = check_det_dataset(str(DATA_YAML), autodownload=False)

for split in ("train", "val"):
    total = labels = 0
    # val은 단일 경로 문자열이다 — 문자열을 그대로 돌면 글자 단위로 쪼개져
    # '/' 한 글자가 전체 파일시스템 glob이 되므로 반드시 목록으로 감싼다.
    dirs = info[split] if isinstance(info[split], list) else [info[split]]
    for p in dirs:
        imgs = glob.glob(str(Path(p) / "**" / "*.*"), recursive=True)
        labels += sum(1 for x in img2label_paths(imgs) if os.path.isfile(x))
        total += len(imgs)
    print(f"{split:>5}: 이미지 {total:>5}  라벨 {labels:>5}  배경 {total - labels:>4}")

print("클래스:", info["names"])

## 6. 베이스 가중치 준비

`models/`는 `.gitignore` 대상이라 비어 있다. 없으면 ultralytics가 받아 온다.

In [ ]:
import shutil
from ultralytics import YOLO

WEIGHTS = REPO / "models" / "yolov8n.pt"
WEIGHTS.parent.mkdir(parents=True, exist_ok=True)

if not WEIGHTS.is_file():
    YOLO("yolov8n.pt")                       # 없으면 현재 폴더로 자동 다운로드
    shutil.move("yolov8n.pt", WEIGHTS)

print(WEIGHTS, "|", WEIGHTS.stat().st_size // 1024, "KB")

## 7. 학습

`output_dir=RUNS`가 이 노트북의 핵심이다 — ultralytics가 `RUNS/custom_detect/weights/last.pt`에
**매 epoch 덮어쓰므로 체크포인트가 Drive에 실시간으로 쌓인다.**

| 인자 | 뜻 |
|---|---|
| `output_dir=RUNS` | 결과를 Drive에 직접 저장 |
| `save_period=5` | 5 epoch마다 `epoch{N}.pt`도 따로 남긴다 (되돌리고 싶을 때) |
| `resume=True` | 체크포인트가 있으면 이어서, 없으면 처음부터 |
| `batch_size=32` | T4(16GB) + yolov8n@640 기준 여유 있는 값. OOM이 나면 16으로 |

> `nbs=64` 때문에 ultralytics가 batch를 64로 정규화해 누적한다. 즉 batch를 16↔32로 바꿔도
> **실효 배치는 64로 같고** 학습 결과는 거의 안 변한다. batch는 속도·메모리 손잡이다.

⚠️ **끊길 것에 대비해 처음에는 `epochs`를 작게(예: 10) 잡고 한 번 끝까지 돌려 보길 권한다.**
파이프라인이 도는 것을 확인한 뒤 늘리는 편이 안전하다.

In [ ]:
from vision_inspection.training import TrainingConfig, train

# 검증 셀(5번)을 건너뛰어도 되도록 여기서 직접 정의한다.
DATA_YAML = REPO / "ros2_ws/src/vision_inspection/config/data.yaml"

config = TrainingConfig(
    data=DATA_YAML,
    base_model=WEIGHTS,
    epochs=50,
    image_size=640,
    batch_size=32,
    run_name="custom_detect",
    output_dir=RUNS,     # ← Drive. last.pt가 매 epoch 여기 쌓인다
    save_period=5,       # 5 epoch마다 스냅샷
    resume=True,         # 끊겼으면 이어서, 처음이면 그냥 시작
)

print("체크포인트:", config.checkpoint_path())
print("이어서 학습:", config.resuming())

train(config)

## 8. 결과 확인

`best.pt`가 최종 산출물이다. 이미 Drive에 있으니 로컬 PC에서 내려받아
`models/`에 두면 `vision_inspection`이 그대로 쓴다.

In [ ]:
from IPython.display import Image as ShowImage, display

run_dir = RUNS / config.run_name
print("결과 폴더:", run_dir)
print("가중치   :", sorted(p.name for p in (run_dir / "weights").glob("*.pt")))

results_png = run_dir / "results.png"
if results_png.is_file():
    display(ShowImage(str(results_png)))

## 9. Hugging Face 업로드 (자동 · 단독 실행 가능)

학습이 끝나면 `best.pt`와 데이터셋 압축을 Hugging Face 저장소로 올린다.
"런타임 → 모두 실행"에 포함되므로 학습이 끝나면 자동으로 올라간다.

**이 셀은 단독 실행이 된다.** 학습 없이 Drive에 이미 있는 결과만 올릴 때는
새 런타임에서 이 셀 하나만 실행하면 된다 — 클론·경로·Drive 마운트를 스스로 한다.

**준비 (한 번만):**

1. [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)에서 **Write** 권한 토큰 발급
2. Colab 왼쪽 **🔑 보안 비밀** 패널에 이름 `HF_TOKEN`으로 저장하고 **"노트북 액세스"를 켠다**
3. 아래 저장소 이름 두 개를 본인 아이디로 바꾼다

- 모델: `best.pt` + `results.png` + `args.yaml` — 매번 새 커밋으로 덮어써 항상 최신 학습을 가리킨다
- 데이터셋: `train_set.tar.gz` — **같은 이름이 이미 있으면 건너뛴다** (바꿨다면 `force=True`)
- 저장소가 없으면 **비공개로 자동 생성**된다
- `RUN_NAME`은 7번 학습 셀의 `run_name`과 같아야 한다

In [ ]:
HF_MODEL_REPO = "<아이디>/<모델-레포>"        # 예: "gildong/smart-factory-ball"
HF_DATASET_REPO = "<아이디>/<데이터셋-레포>"   # 예: "gildong/smart-factory-ball-data"
RUN_NAME = "custom_detect"                   # 7번 학습 셀의 run_name과 같게

# --- 단독 실행 준비. 앞 셀들을 이미 돌렸다면 그냥 지나간다 ---
import sys
from pathlib import Path

REPO = Path("/content/smart-factory-dual-arm")
BRANCH = "feat/yolo"
REMOTE = "https://github.com/yolo-dual-arm-assembly/smart-factory-dual-arm.git"

if REPO.exists():
    !git -C {REPO} fetch --depth 1 origin {BRANCH} && git -C {REPO} reset --hard FETCH_HEAD
else:
    !git clone --branch {BRANCH} --depth 1 {REMOTE} {REPO}

for sub in ("ros2_ws/src/common", "ros2_ws/src/vision_inspection"):
    path = str(REPO / sub)
    if path not in sys.path:
        sys.path.insert(0, path)
for name in [m for m in sys.modules if m.split(".")[0] in ("vision_inspection", "common")]:
    del sys.modules[name]

%pip install -q huggingface_hub

from vision_inspection import colab
from vision_inspection.hf_upload import (
    model_upload_files,
    upload_dataset,
    upload_model,
)

DRIVE = colab.drive_dir()        # 마운트 안 돼 있으면 여기서 인증 팝업이 뜬다
RUNS = colab.drive_runs_dir()

upload_model(HF_MODEL_REPO, model_upload_files(RUNS / RUN_NAME))
upload_dataset(HF_DATASET_REPO, DRIVE / "train_set.tar.gz")   # 바꿨다면 force=True 추가

## 세션이 끊겼다면

1. 이 노트북을 다시 연다 (런타임 유형이 GPU인지 확인).
2. **1번 셀부터 순서대로 다시 실행**한다.
   - 2·4·4b·6번은 이미 있으면 알아서 건너뛴다. (런타임이 초기화됐다면 4b는 COCO를 다시 받는다 — 1~2분.)
   - 7번 학습 셀은 Drive의 `last.pt`를 찾아 **이어서** 시작한다.

### 알아 둘 것

- `resume=True`는 **원래 지정한 `epochs`까지** 채운다. 50 epoch를 이미 다 돌았다면 이어서 할 게 없다.
  더 돌리려면 `epochs`를 늘리는 게 아니라 `run_name`을 바꿔 새로 시작하는 편이 깔끔하다.
- `resume` 없이 같은 `run_name`으로 다시 돌리면 ultralytics가 `custom_detect2`를 새로 만든다.
  진행분을 잃지 않으려는 동작이지만, **이어서 하려던 거라면 `resume=True`를 반드시 켜야 한다.**
- Colab 무료 등급은 연속 실행 시간이 제한된다. `save_period` 스냅샷과 Drive 저장이 있으니
  중간에 끊겨도 잃는 것은 마지막 epoch 하나뿐이다.